# 03: Converting MRI DICOMs to BIDS with HeuDiConv

This notebook converts raw MRI DICOMs into a [BIDS](https://bids.neuroimaging.io/) dataset using [HeuDiConv](https://heudiconv.readthedocs.io/).

**The workflow (two passes):**

1. **Discovery pass** – run HeuDiConv with the `convertall` heuristic and `-c none`. Nothing is converted; it just writes `dicominfo.tsv`, a table of every series in your scan.
2. **Look at the table, write a heuristic** – a small Python file that says "series named *mprage* → `anat/..._T1w`", etc.
3. **Conversion pass** – run HeuDiConv again with your heuristic to produce NIfTI + JSON sidecars + BIDS files.
4. **QC + deface** – inspect the output, then make **defaced copies** of the T1w/T2w with pydeface in a separate `defaced_anat` folder (for sharing), and check the before/after images. The originals stay as they are, because MEG processing (coregistration, FreeSurfer, source localization) needs the full head.
5. **Validate** – run the BIDS validator on the staging folder as a check of the MRI conversion. Notebook 04 then copies the `anat` folders (with the original, non-defaced images) into your main BIDS dataset, next to each session's `meg` folder, with the MRI dates anonymized the same way as the MEG's.

> 🖥️ **Where to run this:** run the notebook *on the server*, since the DICOMs and the Singularity image live there.
>
> 📦 **Get the singularity images once (pin a version so your conversions are reproducible):**
>
> If you don't have these containers, please create folders for them and install with these commands.
>
> Create a hidden cache directory in your own partition, rather than the default home folder location, plus a folder for each container:
>
> ```bash
> export APPTAINER_CACHEDIR=/jukebox/[yourpath]/singularity-containers/cache
> mkdir -p $APPTAINER_CACHEDIR
> mkdir -p /jukebox/[yourpath]/singularity-containers/{heudiconv,pydeface,bids-validator}
> ```

>
> Pull the following containers if you don't already have them:
> 
> ```bash
> apptainer pull /jukebox/[yourpath]/singularity-containers/heudiconv/heudiconv-1.5.1.sif \
>     docker://nipy/heudiconv:1.5.1
>
> apptainer pull /jukebox/[yourpath]/singularity-containers/pydeface/pydeface-2026-08-10.sif \
>     docker://poldracklab/pydeface@sha256:c648b8d70f31a186ff003ed2ad15d491d7b4a7d9adaa4665cf95526415fe3773
> ```
>

> For BIDS Validator, if you already installed it via pip, you can uninstall it first, and then pull the container:
>
> ```bash
> python -m pip uninstall bids-validator-deno
> apptainer pull /jukebox/[yourpath]/singularity-containers/bids-validator/bids-validator-3.0.2.sif \
>     docker://bids/validator:3.0.2
> ```

> ⚠️ **PHI warning:** DICOM headers contain identifying info. HeuDiConv keeps copies of headers in `<output>/.heudiconv/`. Don't push that folder to GitHub / share it without checking.

## 0. Setup

### 0.1 Install the Python packages this notebook uses

This notebook runs in the same `pygers-meg` environment as the earlier notebooks, which already has `pandas` and `matplotlib`. It needs two more packages:

* **pydicom** reads DICOM headers (Step 2)
* **nibabel** loads the NIfTI images for the defacing check (Step 7)

Activate your environment and install them:

```bash
conda activate pygers-meg
pip install pydicom nibabel
```

Check they installed (this should print `ok`):

```bash
python -c "import pydicom, nibabel; print('ok')"
```

If Jupyter was already running, restart the kernel (**Kernel → Restart Kernel**) so it picks up the new packages.

### Running this notebook from the pipeline

**Run the two cells below first, also when you run the notebook yourself.** They define `from_config()`,
which the settings cells use. Run by hand, `config_file` stays `None` and they change nothing: the notebook
uses the settings you edit in its cells. They let the Pygers-MEG-to-BIDS pipeline (`run_bids-pipeline.sh`) run this notebook
with the settings from a study's YAML files instead.

In pipeline mode, each settings cell ends with a line like
`EVENT_ID = from_config("EVENT_ID")`, which replaces the value typed above it with the one from the
YAML files. Run by hand, that line leaves the value as you typed it.

In [ ]:
# Left as None when you run the notebook yourself. The pipeline replaces it with the path of its
# settings file (papermill adds a cell right below this one that sets it).
config_file = None

In [ ]:
# Nothing to edit here. In pipeline mode (config_file set above), this reads this notebook's settings
# from the pipeline's settings file. Run by hand, PIPELINE stays None and from_config() changes nothing.
from pathlib import Path   # file paths that work on macOS, Linux and Windows

NOTEBOOK_ID = "03"     # this notebook's section in the pipeline's settings file
PIPELINE = None            # None = running by hand
if config_file is not None:
    import yaml                                       # reads .yaml files
    with open(config_file, encoding="utf-8") as f:
        PIPELINE = yaml.safe_load(f)[NOTEBOOK_ID]     # a dict: setting name -> value
    print(f"Pipeline mode: notebook {NOTEBOOK_ID} settings from {config_file}")


def from_config(*names):
    """The named settings: from the pipeline's settings file in pipeline mode, unchanged otherwise.

    Example:  EVENT_ID, DROP_UNMAPPED = from_config("EVENT_ID", "DROP_UNMAPPED")

    In pipeline mode every name must be in the settings file: a missing one stops the notebook, so a
    setting can never silently keep this notebook's example value. Settings the file lists under
    "_paths" are turned into Paths.
    """
    if PIPELINE is None:
        values = [globals()[name] for name in names]          # run by hand: keep the values typed above
    else:
        missing = [name for name in names if name not in PIPELINE]
        if missing:
            raise KeyError(f"The pipeline settings file has no value for {missing} (notebook {NOTEBOOK_ID}).")
        paths = set(PIPELINE.get("_paths", []))
        values = [Path(PIPELINE[name]) if name in paths and PIPELINE[name] is not None else PIPELINE[name]
                  for name in names]
    return values[0] if len(values) == 1 else tuple(values)

### 0.2 Configure paths and subjects

Edit the cell below, then run everything top to bottom. Only this cell and the heuristic rules in Step 4 should need changes.

**`DICOM_TEMPLATE`** tells HeuDiConv how your DICOM folders are laid out, relative to `DICOM_ROOT`. Use `{subject}` (and `{session}` if you have sessions) as placeholders. Examples:

| Your folder layout | `DICOM_TEMPLATE` |
|---|---|
| `raw/sub01/SeriesA/img001.dcm` | `{subject}/*/*` |
| `raw/sub01/ses1/SeriesA/img001.dcm` | `{subject}/{session}/*/*` |
| `raw/sub01/*.dcm` (flat) | `{subject}/*` |

In [ ]:
from pathlib import Path

# ---- EDIT THESE LINES -------------------------------------------------------

# ---- Paths -----------------------------------------------------------------
DICOM_ROOT = Path("/jukebox/[yourpath]/auditory-oddball/sourcedata")      # your project's sourcedata/ folder (one sub-* folder per subject)
BIDS_OUT   = Path("/jukebox/[yourpath]/auditory-oddball/sourcedata/sub-01/ses-01/mri-staging")    # STAGING folder for the conversion, next to raw/ (created if missing)

# Where the heuristic file (the rules that map your scans to BIDS names) will be saved.
# Any folder you can write to works; it is created for you in Step 4 if it doesn't exist.
# Keeping it with your study's other code makes it easy to find and reuse later.
# Tip: you can later copy it into your BIDS dataset's code/ folder, which BIDS allows, so the
# dataset carries the rules that named its MRI files (good for reproducibility).
HEURISTIC_PATH = Path("/jukebox/[yourpath]/auditory-oddball/sourcedata/code/heuristic.py")

# Where Step 7 saves the DEFACED copies of the T1w/T2w (same BIDS folder layout and file names).
# These are the versions to use when sharing data outside the lab. The originals (with the
# face) stay in BIDS_OUT; notebook 04 copies them into the main dataset, because MEG processing
# needs the full head.
DEFACED_DIR = Path("/jukebox/[yourpath]/auditory-oddball/sourcedata/sub-01/ses-01/defaced_anat")

# ---- Paths to singularity images -----------------------------------------------------------------
CONTAINER_DIR = Path("/jukebox/[yourpath]/singularity-containers")   # same folder you pulled into (see top of notebook)
SIF_PATH      = CONTAINER_DIR / "heudiconv" / "heudiconv-1.5.1.sif"
PYDEFACE_SIF  = CONTAINER_DIR / "pydeface" / "pydeface-2026-08-10.sif"
VALIDATOR_SIF = CONTAINER_DIR / "bids-validator" / "bids-validator-3.0.2.sif"

# ---- Layout of your DICOMs (relative to DICOM_ROOT) -------------------------
DICOM_TEMPLATE = "sub-{subject}/ses-{session}/raw/mri/dcm/*.dcm"   # DICOMs directly in mri/: ".../raw/mri/*.dcm"

# ---- Which subject(s) / session to convert ----------------------------------
# Subject label must match the folder name in DICOM_ROOT; BIDS output labels
# can only contain letters and numbers (no underscores/dashes).
SUBJECTS = ["01"]
SESSION  = "01"        # -> ses-01. Use None if there are no sessions


# Pipeline mode: these come from the YAML files (run by hand: unchanged). There, BIDS_OUT is the
# session's mri-staging/ folder, DEFACED_DIR its defaced_anat/ folder, and SUBJECTS just this subject.
(DICOM_ROOT, BIDS_OUT, HEURISTIC_PATH, DEFACED_DIR, SIF_PATH, PYDEFACE_SIF, VALIDATOR_SIF,
 DICOM_TEMPLATE, SUBJECTS, SESSION) = from_config(
    "DICOM_ROOT", "BIDS_OUT", "HEURISTIC_PATH", "DEFACED_DIR", "SIF_PATH", "PYDEFACE_SIF",
    "VALIDATOR_SIF", "DICOM_TEMPLATE", "SUBJECTS", "SESSION")


# ---- CHECK THE PATHS ABOVE (nothing below here needs editing) ---------------
# This prints a ✅ / ⚠️ / ❌ line for each folder and container image, so you can
# catch typos before running anything heavy.
#   ✅ = found
#   ⚠️ = not found yet, but that's OK (the notebook will create it, or the step is optional)
#   ❌ = missing and needed, so fix the path (or pull the image) before going on

problems = []   # we collect every ❌ here and report them all at the end

def check(label, path, kind="folder", required=True, created_later=False):
    """Print one status line for `path` and remember it if it's a real problem."""
    if path is None:                                      # an optional path set to None
        print(f"⚠️  {label:<14} not set (optional step will be skipped)")
        return
    path = Path(path)
    if "[yourpath]" in str(path):                         # placeholder never filled in
        print(f"❌  {label:<14} still contains [yourpath] -> {path}")
        problems.append(label)
        return
    found = path.is_file() if kind == "file" else path.is_dir()
    if found:
        print(f"✅  {label:<14} {path}")
    elif created_later and path.parent.is_dir():          # e.g. BIDS_OUT: made automatically later
        print(f"⚠️  {label:<14} doesn't exist yet; it will be created -> {path}")
    elif not required:
        print(f"⚠️  {label:<14} not found (only needed for an optional step) -> {path}")
    else:
        print(f"❌  {label:<14} NOT FOUND -> {path}")
        problems.append(label)

print("Data folders:")
check("DICOM_ROOT",    DICOM_ROOT)
check("BIDS_OUT",      BIDS_OUT, created_later=True)
if "[yourpath]" in str(HEURISTIC_PATH):                   # placeholder never filled in
    print(f"❌  {'HEURISTIC_PATH':<14} still contains [yourpath] -> {HEURISTIC_PATH}")
    problems.append("HEURISTIC_PATH")
elif HEURISTIC_PATH.exists():
    print(f"✅  {'HEURISTIC_PATH':<14} {HEURISTIC_PATH}  (will be overwritten in Step 4)")
else:
    print(f"⚠️  {'HEURISTIC_PATH':<14} doesn't exist yet; Step 4 will write it -> {HEURISTIC_PATH}")

check("DEFACED_DIR",   DEFACED_DIR, created_later=True)

print("\nContainer images:")
if PIPELINE is None:                      # (the pipeline gives each image's path directly)
    check("CONTAINER_DIR", CONTAINER_DIR)
check("SIF_PATH",      SIF_PATH,      kind="file")
check("PYDEFACE_SIF",  PYDEFACE_SIF,  kind="file")
check("VALIDATOR_SIF", VALIDATOR_SIF, kind="file")

# Stop here if anything is missing, so later cells don't fail with confusing errors.
if problems:
    raise FileNotFoundError("Fix these before continuing: " + ", ".join(problems) +
                            "  (missing images? see the pull commands at the top of the notebook)")
print("\n🎉 All paths look good. Carry on!")

In [ ]:
# ---- HeuDiConv settings -----------------------------------------------------
# Options that control how HeuDiConv converts and names your files.
# The defaults below work for most studies, so you can usually leave them as they are.
EXTRA_BINDS = []                 # extra folders to mount, e.g. ["/scratch", "/data/shared"]
OVERWRITE = True                # True -> redo conversions that already exist

# Add _run-01, _run-02... to every filename? Leave False if each scan is acquired once
# per session (cleaner names). Set True if a scan is ever repeated, so repeats get numbered.
ADD_RUN = False

# These scans will join a larger MEG BIDS dataset, so do NOT create top-level files
# (dataset_description.json, participants.tsv/json, README, CHANGES). True -> heudiconv "-b notop".
SKIP_TOP_LEVEL = True

# The scans the heuristic should keep in each session, by the end of their BIDS name. The dry run in
# Step 4.1 checks this before anything is converted (None = don't check).
EXPECTED_SCANS = {"T1w": 1, "T2w": 1}

# Pipeline mode: these come from the YAML files (run by hand: unchanged)
EXTRA_BINDS, OVERWRITE, ADD_RUN, SKIP_TOP_LEVEL, EXPECTED_SCANS = from_config(
    "EXTRA_BINDS", "OVERWRITE", "ADD_RUN", "SKIP_TOP_LEVEL", "EXPECTED_SCANS")

## 1. Check your environment

In [ ]:
# ---- Tools used by the rest of the notebook ---------------------------------
import shutil      # find programs on this computer, copy files
import subprocess  # run terminal commands from inside Python


def sh(cmd, check=False):
    """Run a terminal command and show its output live, as it happens.

    cmd   : the command as a list of words, e.g. ["ls", "-l", "/some/folder"]
    check : if True, stop the notebook with an error when the command fails

    Returns the command's exit code: 0 means it worked, anything else means it failed.
    Later cells use it whenever they run a program, e.g. sh(heudiconv_cmd(...)) in Steps 3 and 5,
    pydeface in Step 7 and the BIDS validator in Step 8.
    """
    cmd = [str(word) for word in cmd]          # Path objects -> plain text
    print("$", " ".join(cmd), "\n")            # show the exact command, so you could copy it into a terminal

    # Start the command. stderr (error messages) is merged into stdout so you see everything in order.
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:                   # print each line as soon as the command writes it
        print(line, end="")

    rc = proc.wait()                           # wait for the command to finish; rc = its exit code
    if check and rc != 0:
        raise RuntimeError(f"Command failed with exit code {rc}")
    return rc


# ---- Find Apptainer (the program that runs the .sif containers) ---------------
# shutil.which() returns the full path to a program, or None if it isn't installed.
# Newer clusters call it "apptainer", older ones "singularity"; they work the same.
SING = shutil.which("apptainer") or shutil.which("singularity")

if SING is None:
    raise RuntimeError("Apptainer/Singularity not found. Are you running this notebook on the cluster? "
                       "(It won't work on a laptop.)")
print("✅ Apptainer found:", SING, "\n")

# Ask the HeuDiConv container for its version. This proves the container actually runs,
# and leaves a record in the notebook of which version did your conversion.
sh([SING, "run", "--cleanenv", SIF_PATH, "--version"])

## 2. Peek at your DICOMs

A quick sanity check that the paths/template are right *before* running anything heavy.

In [ ]:
import glob                    # find files that match a pattern like "*.dcm"
from collections import Counter  # count how many files belong to each series

for sub in SUBJECTS:
    # Fill the subject (and session) into the template, then list every matching file
    pattern = DICOM_TEMPLATE.format(subject=sub, session=SESSION or "*")
    files = sorted(glob.glob(str(DICOM_ROOT / pattern)))
    print(f"sub-{sub}: {len(files)} files match  {pattern}")
    if not files:
        print("   -> nothing matched. Check DICOM_ROOT, SUBJECTS and DICOM_TEMPLATE.")
        continue

    try:
        import pydicom
    except ImportError:
        print("   (pydicom isn't installed, so the headers can't be shown. See 'Install the Python packages' at the top.)")
        continue

    # ---- Study-level info: the same for every file, so read it from the first one ----
    ds = pydicom.dcmread(files[0], stop_before_pixels=True)   # header only, skip the image data (fast)
    for tag in ["PatientID", "StudyDate", "Manufacturer", "MagneticFieldStrength"]:
        print(f"   {tag}: {ds.get(tag, 'n/a')}")

    # ---- Series-level info: read every file's header and count files per series ----
    series = Counter()
    for f in files:
        try:
            h = pydicom.dcmread(f, stop_before_pixels=True,
                                specific_tags=["SeriesNumber", "SeriesDescription"])  # only read what we need
        except Exception:
            continue                                   # skip anything that isn't a readable DICOM
        series[(int(h.get("SeriesNumber", 0)), str(h.get("SeriesDescription", "n/a")))] += 1

    # Print one line per series, in the order they were acquired
    print(f"\n   {len(series)} series found:")
    print(f"   {'#':>4}  {'SeriesDescription':<45} files")
    for (num, desc), n in sorted(series.items()):
        print(f"   {num:>4}  {desc:<45} {n:>5}")


### 2.1 🔒 Look at the `PatientID` line

DICOM headers often hold **identifying information** (PHI): names, initials, birth dates, scan dates. In the sample dataset, `PatientID` contains the participant's **initials**. That shouldn't happen: when registering a participant at the scanner, enter a **study code** (e.g. `sub-01`) as the Patient ID, never their name or initials.

What this means as you go through the notebook:

* **The output above now contains PHI.** Do not share this notebook unles you first clear the outputs (**Kernel → Restart Kernel and Clear All Outputs**).
* **The BIDS files are safer.** dcm2niix (used by HeuDiConv) leaves identifying fields such as `PatientID` out of the JSON sidecars by default.
* **HeuDiConv keeps copies of the raw headers** in the hidden `.heudiconv/` folder inside `BIDS_OUT`. Never share or copy that folder; notebook 04 (Section 12) copies only the `anat` folder and leaves it behind.

## 3. Discovery pass → `dicominfo.tsv`

Before HeuDiConv can convert anything, *you* have to tell it which scan is which (that's the heuristic in Step 4). To write those rules, you first need to know what's in the scan session. That's what this step is for.

We run HeuDiConv with two special settings:

* **`-f convertall`**, a built-in heuristic that simply lists every series it finds
* **`-c none`**, which means "don't convert anything"

So **nothing is converted** here. HeuDiConv reads every DICOM header, groups the files into series, and writes a table describing them: `dicominfo.tsv`. It's saved in the hidden folder `BIDS_OUT/.heudiconv/<subject>/info/` (named `dicominfo_ses-<session>.tsv` when you have sessions).

The cell below builds the HeuDiConv command and runs it (it takes a minute or so).

> 💡 The full command is printed before it runs (the line starting with `$`). You can copy it into a terminal to run HeuDiConv by hand.

In [ ]:
def heudiconv_cmd(subject, heuristic, converter, bids=True):
    """Build the full Apptainer command that runs HeuDiConv for one subject.

    subject   : subject label, e.g. "01"
    heuristic : "convertall" for this discovery pass, or the path to your heuristic file (Step 5)
    converter : "none" = only catalog the series (this step); "dcm2niix" = really convert (Step 5)
    bids      : True -> write BIDS-style output (only matters when actually converting)

    This function is defined once here and reused in Step 5 for the real conversion.
    It only BUILDS the command (a list of words); sh() is what actually runs it.
    """
    BIDS_OUT.mkdir(parents=True, exist_ok=True)       # make the staging folder if it doesn't exist yet

    # ---- HeuDiConv's own options ----------------------------------------------
    args = ["-s", subject,          # -s : which subject to process
            "-f", heuristic,        # -f : which heuristic (naming rules) to use
            "-c", converter]        # -c : which converter to use ("none" = don't convert)
    if bids and converter != "none":
        # -b        : write BIDS files (sidecars, scans.tsv, etc.)
        # -b notop  : same, but skip top-level files like dataset_description.json (see SKIP_TOP_LEVEL)
        args += ["-b", "notop"] if SKIP_TOP_LEVEL else ["-b"]
    if SESSION:
        args += ["-ss", SESSION]    # -ss : session label -> ses-01 in the output names
    if OVERWRITE:
        args += ["--overwrite"]     # redo a subject even if it was converted before

    # ---- Folders the container is allowed to see ("mounts" / "binds") -----------
    # A container can't see your files unless you "bind" a real folder to a path inside it.
    # Each entry is (folder on the cluster, path inside the container, read-only "ro" or read-write "rw").
    mounts = [(DICOM_ROOT.resolve(), "/data", "ro"),   # DICOMs: read-only, so they can't be changed by accident
              (BIDS_OUT.resolve(),   "/base", "rw")]   # staging folder: HeuDiConv writes its output here
    if heuristic != "convertall":
        # A custom heuristic file lives outside the container, so bind its folder at /code
        # and point -f at the file's path INSIDE the container.
        HEURISTIC_PATH.parent.mkdir(parents=True, exist_ok=True)
        mounts.append((HEURISTIC_PATH.parent.resolve(), "/code", "ro"))
        args[args.index("-f") + 1] = f"/code/{HEURISTIC_PATH.name}"
    for extra in EXTRA_BINDS:                          # any extra folders from the HeuDiConv settings cell
        mounts.append((Path(extra).resolve(), str(extra), "rw"))

    # -d : where the DICOMs are (as seen inside the container); HeuDiConv fills in {subject}/{session}
    # -o : where to write the output (as seen inside the container)
    tool_args = ["-d", f"/data/{DICOM_TEMPLATE}", "-o", "/base"] + args

    # Turn the mounts into "-B cluster_folder:container_path:mode" pairs
    binds = [x for src, dst, mode in mounts for x in ("-B", f"{src}:{dst}:{mode}")]

    # Final command: apptainer run --cleanenv -B ... -B ... heudiconv.sif <heudiconv options>
    # --cleanenv keeps your own environment settings from leaking into the container
    return [SING, "run", "--cleanenv"] + binds + [str(SIF_PATH)] + tool_args


# ---- Run the discovery pass for each subject ------------------------------------
for sub in SUBJECTS:
    rc = sh(heudiconv_cmd(sub, "convertall", "none"))   # convertall + no converter = catalog only
    if rc == 0:
        print(f"\n✅ sub-{sub}: discovery pass finished")
    else:
        print(f"\n❌ sub-{sub}: HeuDiConv failed (exit code {rc}). Read the messages above; "
              "the Troubleshooting table at the end of the notebook covers common causes.")

### 3.1 Show the series table

When the discovery pass has finished, HeuDiConv has written one `dicominfo.tsv` per subject (and session) inside the hidden `.heudiconv/` folder. The cell below finds those files and displays the most useful columns, with one row per series. A comment next to each column name in the code explains what it means.

> 🔒 The full `dicominfo.tsv` also contains patient information (ID, age, sex). The cell only shows the columns needed to write the heuristic.

In [ ]:
import pandas as pd   # pandas reads tables (like .tsv files) and shows them nicely

# Show wide tables without cutting off columns or rows
pd.set_option("display.width", 250, "display.max_columns", 50, "display.max_rows", 200)

# Find every dicominfo table HeuDiConv wrote (one per subject/session) in the hidden .heudiconv folder.
# "**" means "look in all subfolders".
tsvs = sorted(BIDS_OUT.glob(".heudiconv/**/dicominfo*.tsv"))
print("Found:", *[str(t.relative_to(BIDS_OUT)) for t in tsvs], sep="\n  ")

# The columns that matter for writing the heuristic (the full file has many more).
# NOTE: the full dicominfo.tsv also contains patient info (ID, age, sex), another reason
# the .heudiconv folder must never be shared. We only display the columns below.
cols = ["series_id",           # number + name HeuDiConv uses to refer to each series
        "series_files",        # how many DICOM files in the series (compare with Step 2)
        "protocol_name",       # protocol name chosen at the scanner: usually the best handle
        "series_description",  # series name shown at the scanner (often the same as protocol_name)
        "dim1", "dim2", "dim3",  # image size: x, y, number of slices
        "dim4",                # number of volumes: 1 = a single 3D image (anat); >1 = 4D (BOLD, DWI)
        "TR", "TE",            # repetition time / echo time (in seconds)
        "is_derived",          # True = made by the scanner from other images (usually skip these)
        "image_type"]          # extra scanner labels, e.g. ORIGINAL / DERIVED, MOSAIC

if not tsvs:
    print("No dicominfo.tsv found. Did the discovery pass above finish without errors?")
for t in tsvs:
    info = pd.read_csv(t, sep="\t")                       # read the tab-separated table
    label = t.relative_to(BIDS_OUT / ".heudiconv").parent.parent   # e.g. 001/ses-01
    print(f"\n{label} — {len(info)} series")
    display(info[[c for c in cols if c in info.columns]])  # show only the columns listed above


### 3.2 👀 Read the table above

Each row is one **series** (one scan). Your job now is to decide which rows go into BIDS, and note what makes each one recognisable, so you can write rules for it in Step 4.

| Look at | Why |
|---|---|
| `protocol_name` / `series_description` | Usually the best handle (e.g. `T1_MPRAGE`, `rest_bold`). If your lab names protocols in a BIDS-like way (e.g. `anat-T1w`), matching is easy. |
| `dim4` | `1` = a single 3D image (anatomical, fieldmap); `> 1` = 4D (functional BOLD, diffusion). |
| `series_files` | A run with far fewer files than its repeats was probably stopped early. Decide whether to keep it. |
| `is_derived` / `image_type` | Scanner-made images (reformats, ADC/FA maps, MoCo series) are usually **skipped**. |
| Localizers / scouts | Positioning scans: always skip them. |

Write down the `protocol_name` of each scan you want to keep. Then adjust the rules in the heuristic in the next section.

## 4. Write the heuristic

The **heuristic** is a small Python file of naming rules. HeuDiConv hands it one record per series (one row of the table in Step 3), and the heuristic decides two things:

1. **Keep or skip?** Scouts, scanner-made reformats and report "series" are skipped.
2. **What BIDS name?** e.g. `sub-01/ses-01/anat/sub-01_ses-01_acq-0p8mm_T1w.nii.gz`

For MEG studies, the MRI session is usually just the **anatomical scans** (used for head models and co-registration), so this heuristic only handles **T1w** and **T2w** images.

The cell below writes the heuristic file to `HEURISTIC_PATH`. The only part you may need to edit is the **word lists** at the top of the cell (`T1W_WORDS`, `T2W_WORDS`, `SKIP_WORDS`, `REQUIRE_WORDS`). They are matched against `protocol_name` and `series_description` from the Step 3 table, in lower case. The defaults match the MEG lab's protocol names (e.g. `anat_ses-01_T1w_acq-0.8mm-MPR`).

> 💡 If the protocol name contains `acq-<something>` (like `acq-0.8mm`), it's copied into the BIDS name as `_acq-0p8mm` (BIDS labels can't contain dots).

> 🛠️ **Your own heuristic:** this template handles T1w and T2w only. For anything else (fieldmaps, DWI,
> functional runs), write your own heuristic file and set `HEURISTIC_SOURCE` to its path: it is then used as
> it is, and never overwritten. The notebook also never overwrites a file at `HEURISTIC_PATH` that it didn't write.

In [ ]:
# ---- EDIT THESE: words that identify each scan (matched in lower case) ------------
# Within one list the words act as OR: a scan matches if its name contains ANY of them.
T1W_WORDS  = ["t1w", "mpr"]                     # e.g. anat_ses-01_T1w_acq-0.8mm-MPR
T2W_WORDS  = ["t2w", "t2space"]                 # e.g. anat_ses-01_T2w_acq-0.8mm-T2SPACE
SKIP_WORDS = ["scout", "localizer", "phoenix"]  # positioning scans and scanner reports
# Optional AND filter: a scan is only converted if its name contains ALL of these words.
# Leave empty ([]) to convert every T1w/T2w. Example: ["0.8mm"] keeps only the 0.8 mm scans
# and skips other T1w/T2w resolutions collected in the same session.
REQUIRE_WORDS = ["0.8mm"]

# HEURISTIC_SOURCE: "generate" = write the heuristic below from these word lists (Recommended).
# Or the path to a heuristic you wrote yourself, used as it is, e.g. for fieldmaps, DWI or functional
# runs, which this template doesn't handle:   HEURISTIC_SOURCE = Path("/path/to/my_heuristic.py")
HEURISTIC_SOURCE = "generate"
# ------------------------------------------------------------------------------------

# Pipeline mode: these come from the YAML files (run by hand: unchanged)
T1W_WORDS, T2W_WORDS, SKIP_WORDS, REQUIRE_WORDS, HEURISTIC_SOURCE = from_config(
    "T1W_WORDS", "T2W_WORDS", "SKIP_WORDS", "REQUIRE_WORDS", "HEURISTIC_SOURCE")

# The heuristic file is written from this text (with HEURISTIC_SOURCE = "generate").
# Lines starting with "#" are comments. The word lists above are filled in further down.
HEURISTIC = r"""
# HeuDiConv heuristic: T1w + T2w anatomicals for MEG studies.
# Written by 03_Dicom_to_BIDS_heudiconv.ipynb (Step 4) from its word lists. Edit them there (or, in the
# pipeline, in processing.yaml), not here: this file is rewritten every time.

import re

# ---- Words that identify each scan (matched in lower case), from the notebook ----------
# Within one list the words act as OR: a scan matches if its name contains ANY of them.
T1W_WORDS  = __T1W_WORDS__
T2W_WORDS  = __T2W_WORDS__
SKIP_WORDS = __SKIP_WORDS__
# AND filter: a scan is only converted if its name contains ALL of these words (empty = no filter).
REQUIRE_WORDS = __REQUIRE_WORDS__

# ---- Filled in by the notebook from SESSION and ADD_RUN (don't edit here) ------
SES_DIR = "__SESDIR__"   # "{session}/" if there are sessions, else ""
SES_TAG = "__SESTAG__"   # "_{session}" if there are sessions, else ""
RUN     = "__RUN__"      # "_run-{item:02d}" if ADD_RUN is True, else ""


def create_key(template, outtype=("nii.gz",), annotation_classes=None):
    # HeuDiConv expects each output name in this (template, file types, extra) form.
    return template, outtype, annotation_classes


def acq_tag(name):
    # Copy "acq-0.8mm" from the protocol name into the BIDS name as "_acq-0p8mm".
    # BIDS labels may only contain letters and numbers, so "." becomes "p".
    # The label stops at the first other character, so "acq-hi-res" gives just "_acq-hi".
    m = re.search(r"acq-([0-9a-z.]+)", name)
    if not m:
        # No "acq-" in the name: leave it out. acq is optional in BIDS, so
        # sub-01_ses-01_T1w.nii.gz is a perfectly good name.
        return ""
    return "_acq-" + m.group(1).replace(".", "p")


def anat_key(suffix, name):
    # e.g. sub-{subject}/{session}/anat/sub-{subject}_{session}_acq-0p8mm_T1w
    # HeuDiConv fills in {subject}, {session} and {item} itself.
    return create_key("sub-{subject}/" + SES_DIR + "anat/sub-{subject}" + SES_TAG
                      + acq_tag(name) + RUN + "_" + suffix)


def has(text, words):
    # True if any of the words appears in the text
    return any(w in text for w in words)


def infotodict(seqinfo):
    # HeuDiConv calls this once per subject/session with one record per series.
    # It returns {BIDS name: [series IDs that should get that name]}.
    info = {}
    for s in seqinfo:
        name = f"{s.protocol_name} {s.series_description}".lower()

        # Skip scanner-made images (reformats, maps) and positioning scans / reports
        if s.is_derived or has(name, SKIP_WORDS):
            continue

        # Skip anything missing one of the REQUIRE_WORDS (all() = every word must be there)
        if not all(w in name for w in REQUIRE_WORDS):
            continue

        # Anatomical scans are single 3D volumes (dim4 == 1)
        if s.dim4 == 1 and has(name, T2W_WORDS):
            info.setdefault(anat_key("T2w", name), []).append(s.series_id)
        elif s.dim4 == 1 and has(name, T1W_WORDS):
            info.setdefault(anat_key("T1w", name), []).append(s.series_id)
        else:
            print("Not converted (no rule matches):", s.series_id)

    return info
"""

# ---- Fill in the session/run settings from the config cells --------------------
if SESSION:
    HEURISTIC = HEURISTIC.replace("__SESDIR__", "{session}/").replace("__SESTAG__", "_{session}")
else:
    HEURISTIC = HEURISTIC.replace("__SESDIR__", "").replace("__SESTAG__", "")
# If a scan was repeated in one session (e.g. a second T1w after motion), set ADD_RUN = True
# in the HeuDiConv settings cell, so the repeats get _run-01, _run-02 instead of clashing.
HEURISTIC = HEURISTIC.replace("__RUN__", "_run-{item:02d}" if ADD_RUN else "")

# Fill in the word lists. repr() writes each list as Python code, e.g. ["t1w", "mpr"].
for name, words in [("T1W_WORDS", T1W_WORDS), ("T2W_WORDS", T2W_WORDS),
                    ("SKIP_WORDS", SKIP_WORDS), ("REQUIRE_WORDS", REQUIRE_WORDS)]:
    HEURISTIC = HEURISTIC.replace(f"__{name}__", repr(list(words)))

# ---- Save it (or use your own file) ----------------------------------------------
GENERATED_MARK = "Written by 03_Dicom_to_BIDS_heudiconv.ipynb"   # in the first lines of every generated file
if HEURISTIC_SOURCE != "generate":
    HEURISTIC_PATH = Path(HEURISTIC_SOURCE)                # your own heuristic, used as it is
    if not HEURISTIC_PATH.is_file():
        raise FileNotFoundError(f"HEURISTIC_SOURCE not found: {HEURISTIC_PATH}")
    print("Using your own heuristic, unchanged:", HEURISTIC_PATH)
else:
    # Never overwrite a heuristic someone wrote by hand: a file without the mark wasn't made here.
    if HEURISTIC_PATH.exists() and GENERATED_MARK not in HEURISTIC_PATH.read_text():
        raise FileExistsError(
            f"{HEURISTIC_PATH} exists but wasn't written by this notebook, so it isn't overwritten.\n"
            "To use it, set HEURISTIC_SOURCE to its path; to generate a new one, set HEURISTIC_PATH to another file.")
    HEURISTIC_PATH.parent.mkdir(parents=True, exist_ok=True)   # make the folder if needed
    HEURISTIC_PATH.write_text(HEURISTIC)
    print("✅ Wrote", HEURISTIC_PATH)

### 4.1 Dry-run the heuristic against `dicominfo.tsv`

Before converting anything, check what the heuristic *would* do with each series from the discovery pass. This runs entirely in Python on the table from Step 3, so it's instant and changes nothing.

You should see the T1w and T2w series mapped to BIDS names and everything else marked *skipped*. If a scan you want is skipped, add a word from its `protocol_name` to `T1W_WORDS` or `T2W_WORDS` above and re-run both cells.

In [ ]:
# ---- Tools --------------------------------------------------------------------
import ast               # turns text like "('ORIGINAL', 'PRIMARY')" back into real Python values
import importlib.util    # lets us load a .py file (the heuristic) and call its functions here
from collections import namedtuple   # makes simple records whose fields we can read as s.protocol_name


# ---- 1. Load the heuristic file written in the cell above ----------------------
# This is the very same file HeuDiConv will use in Step 5, so whatever it decides
# here is exactly what will happen during the real conversion.
spec = importlib.util.spec_from_file_location("heuristic", HEURISTIC_PATH)  # point Python at the file
heuristic = importlib.util.module_from_spec(spec)                         # create an empty module
spec.loader.exec_module(heuristic)                                         # run the file to fill it in
# Now heuristic.infotodict(...) is the function from the file.

if not tsvs:   # tsvs = the list of dicominfo tables found in Step 3
    print("No dicominfo.tsv found. Run the discovery pass (Step 3) first.")


# ---- 2. Dry-run it on each subject's table ---------------------------------------
for t in tsvs:

    # 2a. Read the table. fillna("") replaces empty cells with "" so they don't cause errors.
    df = pd.read_csv(t, sep="\t").fillna("")

    # 2b. Turn each row into a "series record" like the ones HeuDiConv itself would pass in.
    #     During a real run HeuDiConv builds these from the DICOM headers; here we rebuild
    #     them from the table instead. A namedtuple lets the heuristic write s.protocol_name,
    #     s.dim4, s.is_derived, etc., just as it will during the real run.
    Row = namedtuple("Row", df.columns.tolist())   # one field per column of the table
    seqinfo = []                                    # will hold one record per series
    for r in df.to_dict("records"):                 # r = one row, as {column name: value}
        # The .tsv file stores everything as text, so convert the two columns the
        # heuristic checks back into the types it expects:
        if isinstance(r.get("image_type"), str) and r["image_type"].startswith("("):
            r["image_type"] = ast.literal_eval(r["image_type"])            # "('ORIGINAL', ...)" -> tuple
        r["is_derived"] = str(r.get("is_derived", "")).lower() == "true"  # "True"/"False" -> True/False
        seqinfo.append(Row(**r))                    # build the record and add it to the list

    # 2c. Ask the heuristic what it would do. It returns {BIDS name: [series IDs]}.
    #     (Any "Not converted (no rule matches)" lines printed here come from the heuristic.)
    plan = heuristic.infotodict(seqinfo)

    # Count the planned outputs by the end of their BIDS name (e.g. T1w), for the check at the end
    planned = {}
    for (template, _, _), series_ids in plan.items():
        suffix = template.rsplit("_", 1)[-1]           # "..._acq-0p8mm_T1w" -> "T1w"
        planned[suffix] = planned.get(suffix, 0) + len(series_ids)

    # 2d. Flip that around to {series ID: BIDS name}, so we can look up each series.
    #     Each key in plan is (name template, file types, extra); we only need the template.
    dest = {}
    for (template, _, _), series_ids in plan.items():
        for sid in series_ids:
            dest[sid] = template

    # 2e. Print every series with its fate: ✅ + BIDS name, or "skipped".
    label = t.relative_to(BIDS_OUT / ".heudiconv").parent.parent   # subject label, e.g. 001
    print(f"\n{label}:")
    for s in seqinfo:
        target = dest.get(s.series_id)              # None if the heuristic skipped this series
        print(f"  {'✅' if target else '–'} {s.series_id:<45} -> {target or 'skipped'}")
    # {subject} and {session} are filled in by HeuDiConv during the real conversion.

    # 2f. Check: does the heuristic keep exactly the scans in EXPECTED_SCANS (HeuDiConv settings)?
    if EXPECTED_SCANS is not None:
        kinds = set(planned) | set(EXPECTED_SCANS)
        wrong = {k: (planned.get(k, 0), EXPECTED_SCANS.get(k, 0)) for k in kinds
                 if planned.get(k, 0) != EXPECTED_SCANS.get(k, 0)}
        if not wrong:
            print(f"  ✅ the heuristic keeps the expected scans: {EXPECTED_SCANS}")
        else:
            problem = "; ".join(f"{k}: {got} kept, {want} expected" for k, (got, want) in sorted(wrong.items()))
            if PIPELINE is not None:   # pipeline mode: stop before converting the wrong scans
                raise ValueError(f"{label}: the heuristic doesn't keep the expected scans ({problem}). "
                                 "Adjust the word lists (or expected_scans) in the YAML files.")
            print(f"  ⚠️ not what EXPECTED_SCANS says ({problem}). Check the word lists before converting.")

## 5. Convert

Now HeuDiConv runs for real. It uses the same `heudiconv_cmd()` from Step 3, with two changes:

| | Step 3 (discovery) | Step 5 (convert) |
|---|---|---|
| `-f` heuristic | `convertall` (just list everything) | **your heuristic** from Step 4 |
| `-c` converter | `none` (convert nothing) | **`dcm2niix`** (DICOM → NIfTI) |

For each series your heuristic kept, HeuDiConv calls **dcm2niix**, which turns the stack of DICOM files into one **NIfTI** image (`.nii.gz`) plus a **JSON sidecar** with the scan parameters (TR, TE, field strength, …). HeuDiConv then gives both files the BIDS names from your heuristic. With the sample data you should end up with:

```
mri-staging/
└── sub-01/
    └── ses-01/
        ├── anat/
        │   ├── sub-01_ses-01_acq-0p8mm_T1w.nii.gz   + .json
        │   └── sub-01_ses-01_acq-0p8mm_T2w.nii.gz   + .json
        └── sub-01_ses-01_scans.tsv                  (list of files + when each was acquired)
```

After converting, the cell also removes two undocumented columns that HeuDiConv adds to `scans.tsv` (`operator` and `randstr`), which would otherwise cause a BIDS validator warning.

It takes a minute or two per subject. Lots of messages scroll past; that's normal. Look for `ERROR` lines, and for the ✅ / ❌ summary at the end.

**Top-level files:** with `SKIP_TOP_LEVEL = True` the command uses `-b notop`, so HeuDiConv **does not** write `dataset_description.json`, `participants.tsv`, `README` or `CHANGES`; your main MEG dataset keeps its own. The `scans.tsv` file is normal BIDS and sits fine alongside the MEG data.

> 🔒 HeuDiConv also updates the hidden `.heudiconv/` folder (copies of the DICOM headers). That's why `BIDS_OUT` is a **staging** folder: convert here, then copy only the `anat` folder into the main dataset (notebook 04, Section 12) and leave `.heudiconv/` behind.
> Remember we will **merge** the _scans.tsv file with the one produced by MNE-BIDS. Don't overwrite. 

In [ ]:
# ---- Did the heuristic change since the last conversion? ------------------------------
# HeuDiConv keeps its conversion plan in the hidden .heudiconv/ folder. If the heuristic changed, each
# subject's cached plan is cleared, so the new rules are used. A copy of the heuristic used is kept in
# .heudiconv/heuristic_used.py for the next comparison.
USED_HEURISTIC = BIDS_OUT / ".heudiconv" / "heuristic_used.py"
if USED_HEURISTIC.exists() and USED_HEURISTIC.read_text() != HEURISTIC_PATH.read_text():
    for sub in SUBJECTS:
        cache = BIDS_OUT / ".heudiconv" / sub
        if cache.exists():
            shutil.rmtree(cache)               # delete the folder and everything in it
            print(f"The heuristic changed since the last conversion: cleared {cache}")

failed = []   # subjects whose conversion failed
for sub in SUBJECTS:
    # Same command builder as Step 3, but now with YOUR heuristic file and the dcm2niix converter.
    # (Because OVERWRITE = True, re-running this cell redoes the conversion, e.g. after you
    #  edit the heuristic.)
    rc = sh(heudiconv_cmd(sub, str(HEURISTIC_PATH), "dcm2niix"))

    # ---- Quick check: did it produce NIfTI files for this subject? ----------------
    niftis = sorted((BIDS_OUT / f"sub-{sub}").rglob("*.nii.gz"))   # search all subfolders
    if rc == 0 and niftis:
        print(f"\n✅ sub-{sub}: {len(niftis)} NIfTI file(s) written:")
        for f in niftis:
            print("    ", f.relative_to(BIDS_OUT))                 # path inside the staging folder
    elif rc == 0:
        print(f"\n⚠️ sub-{sub}: HeuDiConv finished but wrote no NIfTI files. "
              "Check the Step 4 dry-run: did your heuristic keep any series?")
        failed.append(sub)
    else:
        print(f"\n❌ sub-{sub}: HeuDiConv failed (exit code {rc}). Look for ERROR lines above; "
              "the Troubleshooting table at the end covers common causes.")
        failed.append(sub)

if failed and PIPELINE is not None:   # pipeline mode: stop here, so later steps don't use missing files
    raise RuntimeError(f"HeuDiConv failed or wrote no NIfTI files for: {failed} (see above).")
if not failed:
    shutil.copy2(HEURISTIC_PATH, USED_HEURISTIC)   # remember which heuristic made these files


# ---- Tidy up scans.tsv ------------------------------------------------------------
# HeuDiConv adds two extra columns to each sub-XX_ses-XX_scans.tsv: "operator" (always n/a)
# and "randstr" (a random string). They aren't described anywhere, so the BIDS validator
# warns about them (TSV_ADDITIONAL_COLUMNS_UNDEFINED). Keep just filename + acq_time.
import pandas as pd
for tsv in sorted(BIDS_OUT.glob("sub-*/**/*_scans.tsv")):
    df = pd.read_csv(tsv, sep="\t", dtype=str, keep_default_na=False)   # read everything as text
    extra = [c for c in ("operator", "randstr") if c in df.columns]
    if extra:
        df.drop(columns=extra).to_csv(tsv, sep="\t", index=False)
        print(f"Removed {', '.join(extra)} from {tsv.relative_to(BIDS_OUT)}")

## 6. Inspect the result

Before defacing, check that the conversion produced what you expected. Two quick looks:

1. **The folder tree:** every scan your heuristic kept should appear as a `.nii.gz` image with a matching `.json` sidecar, under `sub-<label>/ses-<label>/anat/`, plus one `scans.tsv` per session. (The hidden `.heudiconv/` folder is left out of the listing.)
2. **The images themselves:** the middle slice of each image in three directions.

In [ ]:
def tree(root, max_depth=4):
    """Print the folders and files under `root` as an indented tree (like the `tree` command)."""
    root = Path(root)
    for p in sorted(root.rglob("*")):                 # every file and folder, in alphabetical order
        rel = p.relative_to(root)                     # path relative to root, e.g. sub-01/ses-01/anat
        if any(part.startswith(".") for part in rel.parts):
            continue                                  # skip hidden folders like .heudiconv
        if len(rel.parts) > max_depth:
            continue                                  # don't go deeper than max_depth levels
        indent = "  " * (len(rel.parts) - 1)          # two spaces per level
        icon = "📁 " if p.is_dir() else "📄 "
        print(indent + icon + p.name)

tree(BIDS_OUT)
# Expected for each subject: sub-XXX/ses-XX/anat/ with a .nii.gz + .json per scan,
# and sub-XXX_ses-XX_scans.tsv next to the anat/ folder.

### 6.1 Look at the images

A quick visual check that each conversion worked: for every NIfTI file, the cell below shows the **middle slice in all three directions** (sagittal = side view, coronal = front view, axial = top view) and prints the image size.

What to look for:
* The whole head is there and nothing is cut off
* The image isn't upside down, mirrored or squashed (the cell displays every image in the same standard orientation, so the T1w and T2w should look alike)
* No obvious artifacts (heavy motion blur, stripes, signal dropout)

> 🔒 These images still show the participant's **face**; they are defaced in Step 7. Clear this cell's output before saving or sharing the notebook.

In [ ]:
import nibabel as nib              # reads and writes NIfTI (.nii / .nii.gz) images
import numpy as np
import matplotlib.pyplot as plt

niftis = sorted(BIDS_OUT.glob("sub-*/**/*.nii.gz"))   # every converted image in the staging folder

for nii in niftis:
    img = nib.load(nii)

    # Reorient to the standard "RAS" direction (Right, Anterior, Superior) so every image is
    # displayed the same way, no matter how it was oriented in the scanner.
    img = nib.as_closest_canonical(img)
    data = img.get_fdata()                 # the image as a 3D array of numbers (x, y, z)
    vx, vy, vz = img.header.get_zooms()[:3]   # voxel size in mm along x, y, z

    print(f"{nii.name}")
    print(f"   shape: {data.shape}   voxel size (mm): {vx:.2f} x {vy:.2f} x {vz:.2f}")

    # Middle slice in each direction. ".T" (transpose) puts the axes the right way round
    # for plotting; origin="lower" in imshow() keeps the top of the head at the top.
    cx, cy, cz = [n // 2 for n in data.shape[:3]]
    views = [
        ("sagittal (side)",  data[cx, :, :].T, vz / vy),   # y (front-back) across, z (up-down) up
        ("coronal (front)",  data[:, cy, :].T, vz / vx),   # x (left-right) across, z up
        ("axial (top)",      data[:, :, cz].T, vy / vx),   # x across, y up (front of head at top)
    ]

    # Brightness limit: ignore the brightest 0.5% of voxels so a few very bright spots
    # don't make the rest of the image look dark.
    vmax = np.percentile(data, 99.5)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for ax, (title, slice_2d, aspect) in zip(axes, views):
        # aspect = voxel height / width, so non-cubic voxels aren't drawn squashed
        ax.imshow(slice_2d, cmap="gray", origin="lower", vmax=vmax, aspect=aspect)
        ax.set_title(title)
        ax.axis("off")                    # hide the axis ticks and numbers
    fig.suptitle(nii.name, fontsize=10)
    plt.tight_layout()
    plt.show()

if not niftis:
    print("No NIfTI files found. Did the conversion in Step 5 finish?")


## 7. Make defaced copies of the anatomical images

T1w/T2w scans show the participant's **face**, which is identifying. But MEG analysis needs the full head: coregistration with the head digitization, FreeSurfer, and source localization all use the original images. So we **keep the originals** in the staging folder (notebook 04 copies them into your main dataset) and make **defaced copies** for any time the data are shared outside the lab.

This step uses [**pydeface**](https://github.com/poldracklab/pydeface), which lines each image up with a template (using FSL's FLIRT) and zeroes out the face. FSL is already inside the pydeface container you pulled at the top of the notebook.

How it works here:
* **Originals are never changed.** pydeface reads each image from `BIDS_OUT` (mounted read-only) and writes the defaced copy to `DEFACED_DIR`, with the same BIDS folders and file name. The matching `.json` sidecar is copied alongside, so `DEFACED_DIR` can stand in for the `anat/` files when you share.
* **Each image is defaced on its own.** The T1w and T2w have different fields of view, so one mask can't safely be reused for both.
* **Re-running is safe.** Images that already have a defaced copy are skipped (set `REDEFACE = True` to redo them).
* **Always look at the before/after images below.** Defacing occasionally removes brain too, especially on T2w. If that happens, set `REDEFACE = True` and try pydeface's `--cost corratio` option.

In [ ]:
# ---- Defacing settings ----------------------------------------------------------
DEFACE_SUFFIXES = ["T1w", "T2w"]   # which images to deface (the end of the BIDS file name)
REDEFACE        = False            # True -> deface again even if a defaced copy already exists
DEFACE_SUFFIXES, REDEFACE = from_config("DEFACE_SUFFIXES", "REDEFACE")   # pipeline mode: from the YAML files

print("Originals are read from:      ", BIDS_OUT)
print("Defaced copies are written to:", DEFACED_DIR)

In [ ]:
# ---- The pydeface command --------------------------------------------------------
# The pinned pydeface image starts pydeface by itself, so we use "apptainer run"
# followed by pydeface's own arguments.

def pydeface_cmd(infile, outfile):
    """Build the command that reads `infile` and writes its defaced copy to `outfile`."""
    binds = ["-B", f"{infile.parent.resolve()}:/in:ro",    # original's folder: READ-ONLY, can't be changed
             "-B", f"{outfile.parent.resolve()}:/out:rw"]  # DEFACED_DIR folder: pydeface writes here
    pydeface_args = [f"/in/{infile.name}",                 # input image (path inside the container)
                     "--outfile", f"/out/{outfile.name}",  # where to save the defaced copy
                     "--force"]                            # allow overwriting (only happens with REDEFACE)
    # --cleanenv keeps your own environment settings from leaking into the container
    return [SING, "run", "--cleanenv"] + binds + [str(PYDEFACE_SIF)] + pydeface_args


# ---- Find the images to deface --------------------------------------------------
targets = []
for sub in SUBJECTS:
    for suf in DEFACE_SUFFIXES:
        # e.g. every sub-01/**/anat/*_T1w.nii.gz in the staging folder
        targets += sorted((BIDS_OUT / f"sub-{sub}").rglob(f"anat/*_{suf}.nii.gz"))
print(f"{len(targets)} image(s) to deface\n")


# ---- Deface each one -------------------------------------------------------------
deface_failed = []   # images pydeface couldn't deface
for f in targets:
    rel = f.relative_to(BIDS_OUT)          # e.g. sub-01/ses-01/anat/sub-01_ses-01_..._T1w.nii.gz
    out = DEFACED_DIR / rel                # same path, but inside DEFACED_DIR

    if out.exists() and not REDEFACE:
        print("already defaced, skipping:", rel)
        continue

    out.parent.mkdir(parents=True, exist_ok=True)      # make the sub-/ses-/anat folders
    rc = sh(pydeface_cmd(f, out))                      # run pydeface (takes a few minutes)

    if rc != 0 or not out.exists():
        print(f"\n❌ FAILED: {rel}  (the original is untouched)\n")
        out.unlink(missing_ok=True)                    # remove any half-written file
        deface_failed.append(str(rel))
        continue

    # Copy the JSON sidecar too, so DEFACED_DIR has complete BIDS file pairs
    sidecar = f.with_name(f.name.replace(".nii.gz", ".json"))
    if sidecar.exists():
        sidecar_out = out.with_name(sidecar.name)
        # HeuDiConv makes its files read-only, and copy2 copies that too: remove an earlier copy first
        # (it can't be overwritten), then make the new copy writable for you (the owner).
        sidecar_out.unlink(missing_ok=True)
        shutil.copy2(sidecar, sidecar_out)
        sidecar_out.chmod(sidecar_out.stat().st_mode | 0o200)   # 0o200 = write permission for the owner
    print(f"\n✅ DEFACED: {rel}\n")

if deface_failed and PIPELINE is not None:   # pipeline mode: a failed defacing stops the run
    raise RuntimeError(f"pydeface failed for: {deface_failed} (the originals are untouched; see above).")

In [ ]:
# ---- Before/after check: original (left) vs defaced copy (right) ------------------
# For each image: the middle sagittal slice (side view, best for seeing the face) and an
# axial slice low in the head (through the nose/eyes). The face should be gone in the
# defaced copy, and NO brain should be missing.
# 🔒 The "original" panels show the face: clear this cell's output before sharing the notebook.
try:
    import nibabel as nib, numpy as np, matplotlib.pyplot as plt
except ImportError:
    print("nibabel isn't installed. See 'Install the Python packages' at the top, then restart the kernel and re-run this cell.")
else:
    # Pair each original with its defaced copy (only where the copy exists)
    pairs = [(f, DEFACED_DIR / f.relative_to(BIDS_OUT)) for f in targets
             if (DEFACED_DIR / f.relative_to(BIDS_OUT)).exists()]
    if not pairs:
        print("Nothing to show yet. Run the defacing cell above first.")
    else:
        # One row per image; 4 columns: original sag, original axial, defaced sag, defaced axial
        fig, axes = plt.subplots(len(pairs), 4, figsize=(16, 4 * len(pairs)), squeeze=False)
        for row, (orig, new) in enumerate(pairs):
            for col, (path, label) in enumerate([(orig, "original"), (new, "defaced")]):
                # Same standard orientation as Step 6, so both versions line up
                data = nib.as_closest_canonical(nib.load(path)).get_fdata()
                vmax = np.percentile(data, 99.5)                  # brightness limit
                sag = data[data.shape[0] // 2, :, :].T            # middle sagittal slice
                axi = data[:, :, int(data.shape[2] * 0.35)].T     # axial slice 35% up from the bottom
                for k, (sl, view) in enumerate([(sag, "sagittal"), (axi, "axial")]):
                    ax = axes[row, col * 2 + k]
                    ax.imshow(sl, cmap="gray", origin="lower", vmax=vmax)
                    ax.set_title(f"{label} - {view}\n{orig.name[:42]}", fontsize=8)
                    ax.axis("off")
        plt.tight_layout()
        plt.show()

## 8. Validate the staging folder with the BIDS validator

The [BIDS validator](https://bids-standard.github.io/bids-validator/) checks that the files follow the BIDS rules: folder layout, file names and required metadata. Here we run it on the **staging folder**, as a check that the MRI conversion itself is sound.

The staging folder isn't a complete dataset (it deliberately has no top-level files), so expect these messages. They are **fine for staging**:

| Message | Why it's OK here |
|---|---|
| `[ERROR] MISSING_DATASET_DESCRIPTION` | Staging has no `dataset_description.json` by design (`SKIP_TOP_LEVEL`). Your main dataset has its own (mne-bids writes it with the MEG data). |
| `[WARNING] SIDECAR_KEY_RECOMMENDED` (e.g. `SpoilingType`) | Optional ("recommended") metadata the scanner doesn't record. Nice to have, not required. |

**Anything else** (another `[ERROR]`, or warnings about file names or missing sidecars) points to a real problem with the conversion and is worth fixing before you go on.

> 📌 This is only a check of the MRI part. Validate the **whole** dataset again once the MEG, eye-tracking and MRI data are all in the main BIDS folder (notebook 06, `06_Validate_BIDS.ipynb`).

In [ ]:
# Run the validator container on the staging folder. The folder is mounted read-only
# at /data, so the validator can look at the files but never change them.
# Expected for staging: 1 error (MISSING_DATASET_DESCRIPTION) + the warnings in the table above.
sh([SING, "run", "--cleanenv",
    "-B", f"{BIDS_OUT.resolve()}:/data:ro",
    VALIDATOR_SIF, "/data"])


## 9. Next: add the MRI to your main dataset (notebook 04)

The last step, copying each session's `anat` folder from this staging folder into your main BIDS dataset
(next to `meg/`), is done by **notebook 04, Section 12**, not here. Two reasons:

* **The MRI dates must be anonymized the same way as the MEG's.** HeuDiConv's `scans.tsv` holds the real
  acquisition date and time. Notebook 04 moves the MEG dates back by `DAYSBACK` days, and it moves the MRI
  times back by the same number of days, so the dataset holds no real dates and the time between the MRI
  and the MEG is kept.
* **The main dataset is created by notebook 04.** MNE-BIDS writes it, including each session's
  `scans.tsv`, with the MEG data; the MRI rows are then added to that file.

Notebook 04 copies only the `anat` folder (the original, non-defaced images), leaves `.heudiconv/` behind,
and converts HeuDiConv's local scanner times to UTC (its `SCANNER_TIMEZONE` setting). Point its
`MRI_STAGING` setting at this notebook's `BIDS_OUT`, and use the same subject and session labels.

> 🔒 The main dataset will then contain identifiable images (the face). Keep it on lab storage with the
> same access controls as the raw DICOMs. When you share the dataset outside the lab (e.g. on OpenNeuro),
> use the defaced copies from `DEFACED_DIR` in place of the `anat/` images.

## Troubleshooting

| Problem | What to do |
|---|---|
| Step 2 says **`0 files match`** | Check `DICOM_ROOT`, `DICOM_TEMPLATE`, and that each name in `SUBJECTS` matches its folder name exactly. |
| A scan you want shows **skipped** in the Step 4.1 dry-run | Add a word from its `protocol_name` (lower case) to `T1W_WORDS` or `T2W_WORDS`, then re-run Step 4 and 4.1. |
| The same scan was acquired **twice** in a session (e.g. a repeated T1w) | Set `ADD_RUN = True` so the repeats get `_run-01`, `_run-02`, or use `REQUIRE_WORDS` to keep only the one you want. |
| You changed the heuristic but the output didn't change | Make sure `OVERWRITE = True`, then re-run Step 5. |
| Defacing removed part of the **brain** (Step 7 before/after images) | The original is untouched. Set `REDEFACE = True`, add `"--cost", "corratio"` to `pydeface_args` in the Step 7 cell, and re-run. |
| A container can't see a folder (`No such file or directory` for a path that exists) | Add the folder to `EXTRA_BINDS` in the HeuDiConv settings cell. |

**Next:** notebook 04 converts the MEG recordings to BIDS and copies this `anat` folder into the dataset (Section 12). Notebook 05 syncs and adds the eye-tracking data converted in notebook 02. Notebook 06 then validates the whole dataset.